# Import thư viện và Đọc dữ liệu gốc

In [ ]:
#Khai báo thư viện
import pandas as pd
import numpy as np
#Đọc file và và chuyển dạng chuỗi (string) trước để tự kiểm soát việc ép kiểu
returns='/content/returns.csv'
df_raw = pd.read_csv(returns)
# , dtype=str, keep_default_na=True
df_raw.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


# Xem tổng quan

In [ ]:
#Đọc số dòng cột của file và xem kiểu dữ liệu của từng thuộc tính
df=df_raw.copy()
print("Số dòng và số cột",df.shape)
print("\nTên cột & dtype hiện tại (đọc dạng string để kiểm soát):")
print(df.dtypes)

Số dòng và số cột (39939, 7)

Tên cột & dtype hiện tại (đọc dạng string để kiểm soát):
return_id           object
order_id             int64
product_id           int64
return_date         object
return_reason       object
return_quantity      int64
refund_amount      float64
dtype: object


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  object 
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  object 
 4   return_reason    39939 non-null  object 
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 2.1+ MB


In [ ]:
df.describe()

,order_id,product_id,return_quantity,refund_amount
count,39939.000000,39939.000000,39939.000000,39939.000000
mean,409061.984176,1244.232730,2.743834,12784.458964
std,240063.904576,691.747822,1.828260,14092.150154
min,2.000000,3.000000,1.000000,458.810000
25%,202651.000000,702.000000,1.000000,3573.395000
50%,404254.000000,992.000000,2.000000,7888.880000
75%,615620.000000,2048.000000,4.000000,16881.990000
max,833351.000000,2412.000000,8.000000,160937.940000


# Định nghĩa các hàm dùng chung

In [ ]:
#thống kê và tìm kiếm các giá trị bất thường (outlier) bằng IQR
def iqr_outlier_stats(s):
    s = pd.to_numeric(s, errors='coerce')
    if s.empty:
      return 0, 0.0, None, None
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = s[(s < lo) | (s > hi)]
    # Trả về 4 thông số: Số lượng outlier, tỷ lệ %, ngưỡng dưới và ngưỡng trên.
    return len(outliers), round(len(outliers) / len(s) * 100, 3), lo, hi
#phát hiện các giá trị bất thường (outlier) bằng Z-score
from sklearn.preprocessing import StandardScaler
def get_zscore_outliers(s, thresh=3.0):
    s = pd.to_numeric(s, errors='coerce')
    if s.empty or s.std(ddof=0) == 0: return 0, 0.0
    scaler = StandardScaler()
    z_scores = scaler.fit_transform(s.values.reshape(-1, 1))
    return pd.Series(np.abs(z_scores.flatten()) > thresh, index=s.index)
#index=s.index: gắn lại đúng tên dòng (nhãn chỉ mục) của bảng dữ liệu gốc cho cột kết quả
# flatten(): chuyển mảng 2 chiều thành mảng 1 chiều
# reshape(-1, 1): -1:chọn tất cả giá trị và 1:chuyển về mảng 1 chiều
def normalize_person_name(x):
    """Trim khoảng trắng thừa (kể cả khoảng trắng ở giữa), viết hoa chữ cái đầu mỗi từ."""
    if pd.isna(x):
        return x
    x = " ".join(str(x).strip().split())
    return x.title() if x != "" else np.nan
def normalize_zip_code(x):
    """Chỉ giữ lại các ký tự số và trả về dưới dạng chuỗi (text), không ép độ dài."""
    if pd.isna(x):
        return x
    # Ép kiểu sang str và lọc lấy các ký tự là số
    digits = "".join(ch for ch in str(x).strip() if ch.isdigit())
    if digits == "":
        return np.nan
    # Trả về chuỗi kết quả nguyên bản
    return digits
def normalize_phone_vn(x, expected_len=9):
    """Số điện thoại VN chuẩn 10 số, bắt đầu bằng 0. Nếu nguồn chỉ có 9 số (mất số 0 đầu) thì bù lại."""
    if pd.isna(x):
        return x
    digits = "".join(ch for ch in str(x).strip() if ch.isdigit())
    if digits == "":
        return np.nan
    if len(digits) == expected_len and not digits.startswith("0"):
        digits = "0" + digits
    return digits
# 1. Rút gọn hàm làm sạch tiền tệ
def clean_currency_string(x):
    if pd.isna(x): return np.nan
    s = str(x)
    for tok in ['₫', 'VND', 'vnd', '$', 'USD', 'usd', ',', '%']:
        s = s.replace(tok, '')
    try:
        return float(s.strip())
    except ValueError:
        return np.nan
# Từ điển chữ số (có thể bổ sung thêm nếu cần)
WORD_NUMBER_MAP = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50
}
#Rút gọn hàm chuyển chữ/số thành float (dùng .get() thông minh hơn)
def words_or_number_to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    for tok in ["percent", "%%", "%"]:
        s = s.replace(tok, "")
    s = s.strip()
    # Tra cứu từ điển trước, nếu không có thì ép float, lỗi thì trả về NaN
    val = WORD_NUMBER_MAP.get(s)
    if val is None:
        try:
            val = float(s)
        except ValueError:
            return np.nan
    return val if val >= 0 else np.nan

# Kiểm tra trùng lặp và dữ liệu thiếu

In [ ]:
#Kiểm tra giá trị còn thiếu
df.isna().sum()

,0
return_id,0
order_id,0
product_id,0
return_date,0
return_reason,0
return_quantity,0
refund_amount,0


In [ ]:
#Kiểm tra tỷ lệ thiếu ở từng cột
# Viết một hàm nhỏ để lấy giá trị đầu tiên không bị trống
def lay_gia_tri_dau(cot):
    hop_le = cot.dropna()
    if len(hop_le) > 0:
        return hop_le.iloc[0]  # Lấy giá trị đầu tiên
    return None                # Nếu cột toàn rỗng thì trả về None

# Sau đó dùng hàm này cho bảng
profile_df = pd.DataFrame({
    'Cột': df.columns,
    'Kiểu dữ liệu': df.dtypes.values,
    'Số lượng thiếu': df.isna().sum().values,
    'Tỷ lệ thiếu (%)': (df.isna().mean() * 100).round(2).values,
    'Số giá trị unique': df.nunique().values,
    'Ví dụ mẫu': [lay_gia_tri_dau(df[c]) for c in df.columns]
})
profile_df

,Cột,Kiểu dữ liệu,Số lượng thiếu,Tỷ lệ thiếu (%),Số giá trị unique,Ví dụ mẫu
0,return_id,object,0,0.0,39939,RET-000001
1,order_id,int64,0,0.0,36062,2
2,product_id,int64,0,0.0,1286,609
3,return_date,object,0,0.0,3806,2012-07-25
4,return_reason,object,0,0.0,5,late_delivery
5,return_quantity,int64,0,0.0,8,6
6,refund_amount,float64,0,0.0,39560,52458.01


In [ ]:
#Kiểm tra trùng lặp
n_dup_full = df.duplicated().sum()
n_dup_key = df.duplicated(subset=['return_id']).sum()
print(f"Số dòng trùng lặp hoàn toàn: {n_dup_full}")
print(f"Số dòng trùng theo khoá chính return_id: {n_dup_key}")
if n_dup_key > 0:
    df = df.drop_duplicates(subset=['return_id'], keep='first')
print("Kích thước sau khi loại trùng:", df.shape)

Số dòng trùng lặp hoàn toàn: 0
Số dòng trùng theo khoá chính return_id: 0
Kích thước sau khi loại trùng: (39939, 7)


In [ ]:
cols = df.columns.to_list()
for c in cols:
    if c in df.columns:
        so_luong = (df[c].value_counts().sort_index(ascending=True) > 1).sum()
        print(f"Số lượng {c} xuất hiện nhiều hơn 1 lần là: {so_luong}")


Số lượng return_id xuất hiện nhiều hơn 1 lần là: 0
Số lượng order_id xuất hiện nhiều hơn 1 lần là: 3801
Số lượng product_id xuất hiện nhiều hơn 1 lần là: 1100
Số lượng return_date xuất hiện nhiều hơn 1 lần là: 3745
Số lượng return_reason xuất hiện nhiều hơn 1 lần là: 5
Số lượng return_quantity xuất hiện nhiều hơn 1 lần là: 8
Số lượng refund_amount xuất hiện nhiều hơn 1 lần là: 375


In [ ]:
for c in cols:
    if c in df.columns:
        counts = df[c].value_counts()
        frequent_values = counts[counts > 1].index
        print(f"giá trị của cột {c} xuất hiện nhiều hơn 1 lần là:",list(frequent_values))

giá trị của cột return_id xuất hiện nhiều hơn 1 lần là: []
giá trị của cột order_id xuất hiện nhiều hơn 1 lần là: [162989, 684619, 433619, 319880, 145681, 615419, 689967, 443198, 42251, 454314, 257512, 424747, 409997, 290806, 290812, 52196, 283503, 240513, 500030, 605659, 337279, 349263, 500618, 265312, 702501, 333676, 191080, 468410, 50128, 143235, 795558, 161383, 667670, 44541, 410949, 319108, 232552, 342083, 182588, 351094, 226719, 437430, 182324, 309065, 465077, 420784, 319468, 122874, 135132, 530455, 351188, 78865, 663547, 217934, 165230, 252224, 128731, 521620, 583995, 179217, 41314, 475681, 339720, 27516, 206473, 330928, 451354, 27504, 7622, 41128, 490768, 65261, 455990, 204127, 213331, 277786, 190265, 288938, 757704, 291641, 288920, 190170, 757528, 370461, 671247, 289194, 86936, 578739, 90321, 289579, 87306, 669085, 509166, 189655, 577263, 370359, 367857, 668199, 190737, 370955, 576806, 291734, 86277, 86289, 758375, 511428, 86309, 86324, 579061, 370897, 87400, 668325, 511426, 3

# Chuẩn hoá định dạng & ép kiểu dữ liệu

In [ ]:
# Chuẩn hoá format và chuyển kiểu
df['return_id'] = df['return_id'].astype(str).str.strip()
# Ép kiểu ID và số lượng về số nguyên an toàn (hỗ trợ NaN)
for c in ['order_id', 'product_id', 'return_quantity']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
# Dùng chuẩn hóa ngày tháng
df['return_date'] = pd.to_datetime(df['return_date'], errors='coerce')
# Chuẩn hóa lý do trả hàng (xóa khoảng trắng thừa, viết thường)
df['return_reason'] = df['return_reason'].astype(str).str.strip().str.lower()
# Dùng hàm clean_currency_string để làm sạch tiền tệ (bỏ $, ₫, dấu phẩy...) ép về float
df['refund_amount'] = df['refund_amount'].apply(clean_currency_string)
print("Giá trị return_reason sau chuẩn hoá:")
print(sorted(df['return_reason'].dropna().unique()))
# Hiển thị nhanh một vài dòng kết quả
df[df.columns].head()

Giá trị return_reason sau chuẩn hoá:
['changed_mind', 'defective', 'late_delivery', 'not_as_described', 'wrong_size']


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


# Kiểm tra logic nghiệp vụ cho từng cột

In [ ]:
import datetime

print("\n--- Kiểm tra logic nghiệp vụ cho từng cột ---\n")

# 1. return_id
print("Kiểm tra cột 'return_id':")
if df['return_id'].duplicated().any():
    print("  Lỗi: Có giá trị 'return_id' trùng lặp.")
else:
    print("  'return_id' là duy nhất. (Đã kiểm tra kỹ ở bước trước)")
# Kiểm tra định dạng RET-XXXXXX (nếu cần)
# if not df['return_id'].astype(str).str.match(r'RET-\d{6}').all():
#     print("  Cảnh báo: Một số 'return_id' không theo định dạng 'RET-XXXXXX'.")

# 2. order_id và product_id
print("\nKiểm tra cột 'order_id' và 'product_id':")
for col in ['order_id', 'product_id']:
    if (df[col] <= 0).any():
        print(f"  Lỗi: Cột '{col}' chứa giá trị không hợp lệ (nhỏ hơn hoặc bằng 0).")
    else:
        print(f"  Cột '{col}' chỉ chứa giá trị dương hợp lệ.")

# 3. return_date
print("\nKiểm tra cột 'return_date':")
if df['return_date'].isnull().any():
    print("  Lỗi: Cột 'return_date' chứa giá trị rỗng.")
elif (df['return_date'] > datetime.datetime.now()).any():
    print("  Lỗi: Cột 'return_date' chứa ngày trong tương lai.")
else:
    print("  'return_date' không chứa giá trị rỗng và không có ngày trong tương lai.")

# 4. return_reason
print("\nKiểm tra cột 'return_reason':")
expected_reasons = ['changed_mind', 'defective', 'late_delivery', 'not_as_described', 'wrong_size']
if not df['return_reason'].dropna().isin(expected_reasons).all():
    print("  Lỗi: Cột 'return_reason' chứa giá trị không mong đợi.")
    print("  Các giá trị không mong đợi: ", df[~df['return_reason'].isin(expected_reasons)]['return_reason'].unique().tolist())
else:
    print("  'return_reason' chứa các giá trị hợp lệ.")

# 5. return_quantity
print("\nKiểm tra cột 'return_quantity':")
if (df['return_quantity'].isnull().any()):
    print("  Lỗi: Cột 'return_quantity' chứa giá trị rỗng.")
elif (df['return_quantity'] <= 0).any():
    print("  Lỗi: Cột 'return_quantity' chứa giá trị không hợp lệ (nhỏ hơn hoặc bằng 0).")
elif (df['return_quantity'] > 10).any(): # Giả định số lượng trả hàng hợp lý là không quá 10
    print("  Cảnh báo: Cột 'return_quantity' có giá trị lớn bất thường (>10).")
else:
    print("  'return_quantity' chứa các giá trị hợp lệ (lớn hơn 0 và trong phạm vi hợp lý).")

# 6. refund_amount
print("\nKiểm tra cột 'refund_amount':")
if df['refund_amount'].isnull().any():
    print("  Lỗi: Cột 'refund_amount' chứa giá trị rỗng.")
elif (df['refund_amount'] <= 0).any():
    print("  Lỗi: Cột 'refund_amount' chứa giá trị không hợp lệ (nhỏ hơn hoặc bằng 0).")
else:
    print("  'refund_amount' chứa các giá trị hợp lệ (lớn hơn 0).")


--- Kiểm tra logic nghiệp vụ cho từng cột ---

Kiểm tra cột 'return_id':
  'return_id' là duy nhất. (Đã kiểm tra kỹ ở bước trước)

Kiểm tra cột 'order_id' và 'product_id':
  Cột 'order_id' chỉ chứa giá trị dương hợp lệ.
  Cột 'product_id' chỉ chứa giá trị dương hợp lệ.

Kiểm tra cột 'return_date':
  'return_date' không chứa giá trị rỗng và không có ngày trong tương lai.

Kiểm tra cột 'return_reason':
  'return_reason' chứa các giá trị hợp lệ.

Kiểm tra cột 'return_quantity':
  'return_quantity' chứa các giá trị hợp lệ (lớn hơn 0 và trong phạm vi hợp lý).

Kiểm tra cột 'refund_amount':
  'refund_amount' chứa các giá trị hợp lệ (lớn hơn 0).


# Kiểm tra khóa ngoại

In [ ]:
# Tải bảng orders_enriched
orders_df = pd.read_csv('/content/orders_enriched.csv', dtype=str)
orders_df['order_id'] = pd.to_numeric(orders_df['order_id'], errors='coerce').astype('Int64')
# Tải bảng products
products_df = pd.read_csv('/content/products.csv', dtype=str)
products_df['product_id'] = pd.to_numeric(products_df['product_id'], errors='coerce').astype('Int64')

In [ ]:
# Kiểm tra khóa ngoại cho order_id
orders_not_in_master = df[~df['order_id'].isin(orders_df['order_id'])]
if not orders_not_in_master.empty:
    print(f"Có {len(orders_not_in_master)} `order_id` trong `df` không tồn tại trong `orders_enriched`.")
    print("Các `order_id` không khớp:", orders_not_in_master['order_id'].unique().tolist())
else:
    print("Tất cả `order_id` trong `df` đều khớp với `orders_enriched`.")
# Kiểm tra khóa ngoại cho product_id
products_not_in_master = df[~df['product_id'].isin(products_df['product_id'])]
if not products_not_in_master.empty:
    print(f"\nCó {len(products_not_in_master)} `product_id` trong `df` không tồn tại trong `products`.")
    print("Các `product_id` không khớp:", products_not_in_master['product_id'].unique().tolist())
else:
    print("\nTất cả `product_id` trong `df` đều khớp với `products`.")

Tất cả `order_id` trong `df` đều khớp với `orders_enriched`.

Tất cả `product_id` trong `df` đều khớp với `products`.


# Xuất file kết quả

In [ ]:
df.to_csv('returns_silver.csv', index=False, encoding='utf-8-sig')
print("Đã lưu file đã làm sạch tại: returns_silver.csv",)
print(f"Kích thước cuối cùng: {df.shape}")

Đã lưu file đã làm sạch tại: returns_silver.csv
Kích thước cuối cùng: (39939, 7)
